# Do Chatbot ao Agente: Construindo Sistemas de IA com Ferramentas

Neste notebook você roda, de ponta a ponta, um agente que **raciocina em passos** usando ferramentas:

1. **Modelo** (Gemini, via LangChain), que só *pede* chamadas de ferramenta: quem executa é o nosso código.
2. **Memória de trabalho**: a API é *stateless*, então o histórico completo é reenviado a cada passo.
3. **Registry de ferramentas**: `update_plan` (o agente registra o raciocínio) e `finish` (entrega a resposta e encerra).
4. **Loop**: chama o modelo, executa as ferramentas pedidas e repete até `finish` (ou até `max_steps`).
5. **Tracing**: tokens, latência e plano de cada passo.

**Antes de começar:** no painel **Secrets** (ícone de chave 🔑 à esquerda), crie o secret `GEMINI_API_KEY` com a **sua** chave
do Google AI Studio e **ative "Acesso ao notebook"** para este notebook. Opcional: `GEMINI_MODEL` para trocar o modelo.
Nunca cole a chave numa célula.

## 1. Preparar o ambiente

Clona o repositório (ou atualiza, se já existir), instala as dependências e coloca o pacote no `sys.path`.

> Se você atualizar o repositório **depois** de importar o pacote, o Python continua com a versão antiga na memória:
> reinicie o ambiente de execução (*Ambiente de execução → Reiniciar sessão*) e rode as células de novo
> (ou use `importlib.reload()`).

In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/pedroop32/workshop-agentes.git"
REPO_DIR = "/content/workshop-agentes"


def sh(*cmd: str) -> None:
    """Roda um comando; mostra a saída só se der erro."""
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stdout, result.stderr)
        raise RuntimeError(f"Falhou: {' '.join(cmd)}")


if os.path.isdir(REPO_DIR):
    sh("git", "-C", REPO_DIR, "pull", "--ff-only")
else:
    sh("git", "clone", REPO_URL, REPO_DIR)
sh(sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements.txt")

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print("Ambiente pronto.")

## 2. Configuração e teste de conexão

Lê a chave e o modelo dos Secrets (`get_secret`: Colab primeiro, depois variável de ambiente/`.env`) e faz **uma** chamada
simples para confirmar que tudo funciona. Em caso de falha, mostramos só o **tipo** do erro, nunca a mensagem
(ela poderia expor dados). Se o modelo não for encontrado, listamos os modelos disponíveis para a sua chave.

Também imprimimos `usage_metadata` e `finish_reason`, os dados que o tracing usa.

In [ ]:
from langchain_core.exceptions import ModelNotFoundError

from workshop_agentes.agent.config import DEFAULT_MODEL, AgentConfig, get_secret
from workshop_agentes.llm import build_llm, list_generate_models, text_of

GEMINI_API_KEY = get_secret("GEMINI_API_KEY")
assert GEMINI_API_KEY, "Crie o secret GEMINI_API_KEY (ícone de chave) e ative o acesso do notebook."
MODEL = get_secret("GEMINI_MODEL", DEFAULT_MODEL)
print(f"Modelo: {MODEL}")

config = AgentConfig()  # max_steps=8, temperatura padrão do modelo, verbose=True
llm = build_llm(MODEL, GEMINI_API_KEY, temperature=config.temperature)

try:
    reply = llm.invoke("Responda apenas: conexão funcionando!")
    print(f"SUCESSO: {text_of(reply.content)}")
    print(f"usage_metadata: {reply.usage_metadata}")
    print(f"finish_reason: {reply.response_metadata.get('finish_reason')}")
except Exception as exc:
    print(f"FALHA: {type(exc).__name__}")
    if isinstance(exc, ModelNotFoundError):
        try:
            print("Modelos disponíveis (defina o secret GEMINI_MODEL com um deles):")
            for name in list_generate_models(GEMINI_API_KEY):
                print(" -", name)
        except Exception as list_exc:
            print(f"Não foi possível listar os modelos: {type(list_exc).__name__}")

## 3. Montar o agente

Cada peça é criada separadamente e **injetada** no agente (injeção de dependências): é fácil trocar o modelo,
as ferramentas, a memória ou o tracer sem mexer no loop.

- `update_plan`: o modelo registra o que descobriu, o plano e o próximo passo; o campo `external` aparece ao vivo para você.
- `finish`: ferramenta **terminal**. Quando ela dá certo, o loop termina com a resposta.

In [ ]:
from workshop_agentes.agent.loop import Agent
from workshop_agentes.agent.memory import WorkingMemory
from workshop_agentes.tools.core import finish, update_plan
from workshop_agentes.tools.registry import ToolRegistry
from workshop_agentes.tracing.tracer import Tracer

registry = ToolRegistry()
registry.register(update_plan)
registry.register(finish, terminal=True)

agent = Agent(
    llm=llm,
    registry=registry,
    memory=WorkingMemory(config.system_prompt),
    tracer=Tracer(),
    config=config,
)
print("Ferramentas:", [tool.name for tool in registry.tools])

## 4. Executar o agente

As linhas com `→` são o campo `external` de cada `update_plan`: o que o agente **vai fazer** em seguida.
No fim aparecem a resposta e o **motivo de término**:

| motivo | significado |
|---|---|
| `finish` | o agente chamou `finish` com sucesso |
| `no_tool_call` | o modelo respondeu só com texto |
| `max_steps` | atingiu o limite de passos sem terminar |
| `error` | falha na chamada ao modelo (o tipo do erro aparece em `error`) |

Pode rodar esta célula várias vezes: cada `run()` começa com memória e tracing novos.
Lembre que **cada passo é uma chamada à API** e conta para os limites do plano gratuito.

In [ ]:
TASK = "Compare lista encadeada e array dinâmico para implementar uma fila e conclua qual usar."

result = agent.run(TASK)

print()
print(f"Motivo de término: {result.reason} | passos: {result.steps}")
if result.error:
    print(f"Erro: {result.error}")
print(f"Resposta final:\n{result.answer}")

## 5. Relatório de tracing

Tokens e latência **por passo**, **por função** e no **total**, com o `finish_reason` e o plano registrado em cada passo.
Os totais somam só as chamadas (folhas da árvore de spans), para não contar nada em dobro.

In [ ]:
from workshop_agentes.tracing.report import format_report

print(format_report(result.spans))
print()
print("Invariante da memória (toda chamada tem seu resultado logo depois):", agent.memory.check_invariant())